# PySpark 9: Python UDFs and pandas

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-09-python-udfs-and-pandas.ipynb)

A data scientist hands you a pandas function that scores customers, and a vendor's Python SDK is the only way to
read their API. Spark can run Python code on every executor, but crossing between the JVM and Python has a cost,
and choosing the right kind of Python function can make a job ten times faster or slower.

This lesson shows what happens when Spark calls Python, then works through the options from slowest to most
flexible: row-at-a-time UDFs, Arrow-optimized UDFs, pandas UDFs, `applyInPandas`, `mapInPandas`, Python UDTFs and
Python data sources. It also covers broadcast variables, accumulators and moving data between Spark and pandas.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb) and basic pandas.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pandas pyarrow numpy

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
import pandas as pd
from pyspark.sql import types as T

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

## What happens when Spark calls Python

Built-in functions run inside the executor JVM on Spark's compact binary rows. A Python function cannot: each
executor starts Python worker processes, sends rows to them over a local socket, runs your code, and reads the
results back. That costs serialization in both directions, and the optimizer cannot see inside the function, so
filters do not move past it and code generation stops at it.

In a plan, a Python UDF appears as `BatchEvalPython` (rows pickled in batches) or `ArrowEvalPython` (columns in
Arrow format). Order of preference: a built-in function, a SQL expression or higher-order function, a vectorized
pandas UDF, and only then a row-at-a-time UDF.

**Where it matters**
- **Logic with no built-in equivalent:** a Python library, a model, a vendor SDK.
- **Code reuse:** an existing, tested pandas function.

**Watch out for**
- Most "I need a UDF" cases are built-ins in disguise: `when`, `regexp_extract`, `transform`, `date_format`.
- Python packages your UDF imports must be installed on every executor, not only on the driver.

### Example

Extract an email domain with a built-in expression and with a Python UDF, and compare the plans.

In [ ]:
@F.udf(T.StringType())
def domain_udf(email):
    return email.split("@")[-1].lower() if email and "@" in email else None

emails = spark.createDataFrame([("Ada@Example.COM",), ("grace@example.org",), (None,)], "email string")
builtin = emails.withColumn("domain", F.lower(F.element_at(F.split("email", "@"), -1)))
with_udf = emails.withColumn("domain", domain_udf("email"))

with_udf.show()
with_udf.explain()
print("built-in plan calls Python:", "EvalPython" in plan_of(builtin))

**What to notice:** Both give the same domains, but only the UDF plan contains `BatchEvalPython`: the step that ships rows to a
Python worker and back. The built-in version stays in the JVM. Note also that the UDF had to handle `None`
itself; Python UDFs receive nulls as `None`.

## Write a Python UDF

Decorate a function with `@F.udf(returnType)` to use it on columns, or register it with
`spark.udf.register("name", fn, returnType)` to call it from SQL. Declare the return type: without one, Spark
assumes `string`. Spark assumes UDFs are deterministic and may call them more or fewer times than you expect; mark
a random or time-dependent one with `.asNondeterministic()`.

**Where it matters**
- **Small, row-level logic** with no built-in equivalent, on modest data.
- **SQL users** who need one custom function.

**Watch out for**
- A wrong return type does not fail: values that do not match become null.
- Exceptions inside the UDF fail the whole task; handle bad input in the function.

### Example

Register a UDF for SQL, and see what a wrong return type does.

In [ ]:
def loyalty_tier(lifetime_value):
    if lifetime_value is None:
        return "unknown"
    return "gold" if lifetime_value >= 500 else "silver" if lifetime_value >= 200 else "bronze"

spark.udf.register("loyalty_tier", loyalty_tier, "string")
orders.createOrReplaceTempView("orders")
spark.sql("""
    SELECT customer_id, sum(amount) AS lifetime_value, loyalty_tier(sum(amount)) AS tier
    FROM orders GROUP BY customer_id ORDER BY customer_id
""").show()

wrong_type = F.udf(lambda x: x * 2, T.IntegerType())  # returns a Decimal, declared as int
orders.select("order_id", "amount", wrong_type("amount").alias("doubled")).show(3)

**What to notice:** The registered function works inside SQL like a built-in. The second UDF returns Python `Decimal` values but
declares `int`, so every result is null, with no error at all. Always declare the type the function really
returns.

## Arrow-optimized Python UDFs

Apache Arrow is a columnar in-memory format that both the JVM and Python understand. With `useArrow=True`
(`@F.udf(returnType, useArrow=True)`), Spark sends batches of rows as Arrow data instead of pickling them one by
one, which cuts serialization cost while your function still receives one value at a time.
`spark.sql.execution.pythonUDF.arrow.enabled` turns it on for every UDF.

**Where it matters**
- **Existing row-at-a-time UDFs** you cannot rewrite as pandas UDFs.
- **String-heavy UDFs,** where pickling is a large share of the cost.

**Watch out for**
- Type coercion is stricter with Arrow: return exactly the declared type.
- Your function still runs once per row; vectorizing with a pandas UDF is faster again.

### Example

Run the same function as a pickled UDF and as an Arrow UDF, and compare the plan nodes.

In [ ]:
def clean_name(name):
    return " ".join(part.capitalize() for part in name.split()) if name else None

pickled = F.udf(clean_name, T.StringType())
arrow = F.udf(clean_name, T.StringType(), useArrow=True)
result = customers.select("name", pickled("name").alias("pickled"), arrow("name").alias("arrow"))
result.show(3)
plan = plan_of(result, "simple")
print("BatchEvalPython:", "BatchEvalPython" in plan, "| ArrowEvalPython:", "ArrowEvalPython" in plan)

**What to notice:** Same results, two different plan nodes: `BatchEvalPython` for the pickled UDF and `ArrowEvalPython` for the Arrow
one. On large data the Arrow version spends far less time converting values between the JVM and Python.

## Vectorize with pandas UDFs

A **pandas UDF** (`@F.pandas_udf(returnType)`) receives a whole batch of rows as `pandas.Series` and returns a
Series, so NumPy and pandas do the work in vectorized code. The type hints choose the kind:

- `Series -> Series`: one output value per input row (scalar);
- `Iterator[Series] -> Iterator[Series]`: the same, but with one-time setup per partition (load a model once);
- `Series -> scalar`: an aggregate, usable in `groupBy().agg()` and over windows.

Batches hold up to `spark.sql.execution.arrow.maxRecordsPerBatch` rows (10,000 by default).

**Where it matters**
- **Numerical work** with NumPy, SciPy or pandas.
- **Model scoring** where a model should be loaded once per partition, not once per row.

**Watch out for**
- A Series-to-Series UDF sees one batch at a time: `s.mean()` inside it is the batch mean, not the column mean.
- pandas and PyArrow must be installed on every executor.

### Example

Score orders with a scalar pandas UDF, load a lookup once per partition with an iterator UDF, and compute a grouped median with an aggregate pandas UDF.

In [ ]:
from typing import Iterator


@F.pandas_udf("double")
def log_amount(amount: pd.Series) -> pd.Series:
    import numpy as np
    return np.log1p(amount.astype("float64"))


@F.pandas_udf("string")
def status_label(batches: Iterator[pd.Series]) -> Iterator[pd.Series]:
    labels = {"placed": "Open", "paid": "Paid", "shipped": "On its way", "returned": "Returned"}  # setup once
    for statuses in batches:
        yield statuses.map(labels).fillna("Unknown")


@F.pandas_udf("double")
def median_amount(amount: pd.Series) -> float:
    return float(amount.median())

orders.select("order_id", "amount", F.round(log_amount("amount"), 3).alias("log_amount"),
              status_label("status").alias("label")).show(4)
orders.groupBy("status").agg(median_amount("amount").alias("median_amount")).orderBy("status").show()

**What to notice:** The scalar UDF transformed whole batches with NumPy. The iterator UDF built its lookup dictionary once per
partition and reused it for every batch, which is how you avoid reloading a model per row. The aggregate pandas UDF
received each status's amounts as one Series and returned one value per group, like a built-in aggregate.

## Process whole groups with applyInPandas

`df.groupBy(key).applyInPandas(fn, schema)` gives `fn` all rows of one group as a pandas DataFrame and expects a
pandas DataFrame back, with any number of rows, matching the declared output schema. Spark shuffles the data so
each group arrives in one piece.

**Where it matters**
- **Per-group models or statistics:** fit a trend per store, normalize within each customer.
- **Porting pandas code** that already works group by group.

**Watch out for**
- Each group must fit in one executor's memory; a giant group fails the task.
- Column order and types of the returned DataFrame must match the schema exactly.

### Example

For each customer, compute each order's share of the customer's spend and its rank, in pandas.

In [ ]:
def per_customer(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("amount", ascending=False)
    total = pdf["amount"].astype("float64").sum()
    return pd.DataFrame({
        "customer_id": pdf["customer_id"],
        "order_id": pdf["order_id"],
        "share": (pdf["amount"].astype("float64") / total).round(2),
        "rank_in_customer": range(1, len(pdf) + 1),
    })

shares = orders.groupBy("customer_id").applyInPandas(
    per_customer, schema="customer_id int, order_id int, share double, rank_in_customer int"
)
shares.orderBy("customer_id", "rank_in_customer").show(6)

**What to notice:** Each call received one customer's orders as a pandas DataFrame and returned a new one with the share and rank.
The result matches what a window function would compute ([lesson 7](pyspark-07-aggregations-and-windows.ipynb)); use the window when you can,
and `applyInPandas` when the per-group logic needs pandas or a Python library.

## Transform partitions with mapInPandas

`df.mapInPandas(fn, schema)` passes an iterator of pandas DataFrames (the batches of each partition) to `fn`, which
yields DataFrames of any size. There is no shuffle and no grouping: it is the vectorized equivalent of a flat map,
useful for parsing, filtering or calling an external library on every batch.

**Where it matters**
- **Parsing or enriching records** with a Python library, batch by batch.
- **Expanding or filtering rows** where the output size differs from the input.

**Watch out for**
- Rows from different keys share a batch; do not assume anything about grouping.
- Yield DataFrames with exactly the declared schema.

### Example

Expand each order into one row per day until delivery, a row-count change no built-in `select` can express.

In [ ]:
def delivery_days(batches):
    for pdf in batches:
        days = pdf["status"].map({"shipped": 3, "paid": 5}).fillna(0).astype(int)
        expanded = pdf.loc[pdf.index.repeat(days), ["order_id", "status"]].copy()
        expanded["day_offset"] = expanded.groupby("order_id").cumcount() + 1
        yield expanded

expanded = orders.mapInPandas(delivery_days, schema="order_id int, status string, day_offset long")
expanded.groupBy("status").count().orderBy("status").show()

**What to notice:** Paid orders became five rows each and shipped orders three; placed and returned orders produced none. The number
of output rows changed freely, and no shuffle was needed because each batch was processed on its own.

## Return many rows from a Python UDTF

A **user-defined table function** (UDTF, Spark 3.5 and later) is a Python class whose `eval` method yields any
number of rows per input. Decorate it with `@F.udtf(returnType=...)`, call it like a table in Python, or register it
and use it in SQL with a lateral join.

**Where it matters**
- **Splitting or exploding** values with custom rules.
- **Calling an API** that returns several records per input key.

**Watch out for**
- UDTFs run in Python workers like UDFs: prefer `explode` and built-ins when they can do the job.

### Example

Split a list of tags into rows with a UDTF, and call it from SQL on every customer row.

In [ ]:
@F.udtf(returnType="tag string, position int")
class SplitTags:
    def eval(self, text: str):
        for position, tag in enumerate((text or "").split(",")):
            if tag.strip():
                yield tag.strip().lower(), position

SplitTags(F.lit("Sale, New, ,Outlet")).show()

spark.udtf.register("split_tags", SplitTags)
spark.createDataFrame([(1, "sale,new"), (2, "outlet")], "customer_id int, tags string").createOrReplaceTempView("customer_tags")
spark.sql("SELECT customer_id, tag, position FROM customer_tags, LATERAL split_tags(tags) ORDER BY customer_id, position").show()

**What to notice:** One input string produced three rows; the empty tag between the commas was skipped by the `if`. In SQL, the
lateral join called the UDTF once per customer row and attached its output rows to that customer.

## Share data with executors: broadcast variables and accumulators

A **broadcast variable** (`spark.sparkContext.broadcast(value)`) ships a read-only Python object to each executor
once, instead of with every task. Use `.value` inside a UDF. An **accumulator** is a counter that tasks add to and
only the driver reads, useful for metrics such as "rows that failed to parse".

**Where it matters**
- **Lookup tables inside UDFs:** a dictionary of exchange rates or a small model.
- **Counting events** inside UDFs without a separate aggregation.

**Watch out for**
- For lookups, a broadcast join with a DataFrame is usually simpler and faster than a broadcast variable in a UDF.
- Accumulator updates in transformations can be counted more than once if a task is retried; trust them only for monitoring.
- Neither exists in Spark Connect, which has no `sparkContext`.

### Example

Convert amounts with a broadcast dictionary inside a UDF, and count unknown currencies with an accumulator.

In [ ]:
rates = spark.sparkContext.broadcast({"EUR": 1.0, "USD": 0.92, "GBP": 1.17})
unknown_currency = spark.sparkContext.accumulator(0)


@F.udf("double")
def to_eur(amount, currency):
    rate = rates.value.get(currency)
    if rate is None:
        unknown_currency.add(1)
        return None
    return float(amount) * rate

payments = spark.createDataFrame([(100.0, "USD"), (50.0, "GBP"), (10.0, "JPY"), (20.0, "EUR")], "amount double, currency string")
payments.withColumn("amount_eur", to_eur("amount", "currency")).show()
print("unknown currencies counted:", unknown_currency.value)

**What to notice:** Each executor received the rates dictionary once. The JPY row has no rate, so it became null and the accumulator
counted it; the driver read the count after the action finished. Accumulators are only reliable after an action
and can over-count when tasks are retried.

## Move data between Spark and pandas

`df.toPandas()` collects a DataFrame into a pandas DataFrame on the driver, and `spark.createDataFrame(pdf)` goes
the other way. With `spark.sql.execution.arrow.pyspark.enabled` (on by default in recent versions), both directions
use Arrow and are much faster than row-by-row conversion.

The **pandas API on Spark** (`import pyspark.pandas as ps`) offers pandas-style syntax on distributed data. It is
handy for porting pandas code, but on Spark 4.0 it needs pandas older than 3.0, so it is not run here.

**Where it matters**
- **Plotting and reporting** on an aggregated result.
- **Loading small reference data** prepared in pandas.

**Watch out for**
- `toPandas()` loads everything into driver memory. Aggregate or sample first.
- pandas has no `decimal` type; decimals become Python `Decimal` objects, and timestamps follow the session time zone.

### Example

Aggregate in Spark, convert the small result to pandas, and create a Spark DataFrame from pandas.

In [ ]:
summary = orders.groupBy("status").agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue")).toPandas()
print(type(summary).__name__)
print(summary.sort_values("status").to_string(index=False))
print(summary.dtypes.to_dict())

targets = pd.DataFrame({"status": ["paid", "shipped"], "target": [800.0, 1500.0]})
spark.createDataFrame(targets).printSchema()

**What to notice:** Only the four aggregated rows crossed to the driver. The `decimal` revenue arrived in pandas as `object` dtype
(Python `Decimal` values), because pandas has no decimal type; convert explicitly if you need floats. In the other
direction, Spark inferred `string` and `double` from the pandas dtypes.

## Write a connector with the Python Data Source API

Spark 4 lets you write a custom source or sink in pure Python. Subclass `DataSource` (its name and schema) and
`DataSourceReader` (one `read` call per partition); each `InputPartition` becomes one task. Register the class and
use it like any built-in format: `spark.read.format("name").load()`. Writers and streaming readers follow the same
pattern.

**Where it matters**
- **Internal REST APIs or vendor SDKs** with no maintained Spark connector.
- **Test sources** that generate data for a pipeline.

**Watch out for**
- The classes are pickled and run on executors: import heavy libraries inside `read` and never put credentials in the class body.
- A maintained JDBC, Kafka or file connector is faster and better tested; write your own only when none exists.

### Example

A source that returns one exchange rate per currency, with one partition (and one task) per currency.

In [ ]:
from pyspark.sql.datasource import DataSource, DataSourceReader, InputPartition


class FxRatesDataSource(DataSource):
    @classmethod
    def name(cls):
        return "fx_rates"

    def schema(self):
        return "currency string, rate_to_eur double"

    def reader(self, schema):
        return FxRatesReader()


class FxRatesReader(DataSourceReader):
    def partitions(self):
        return [InputPartition(currency) for currency in ("USD", "GBP", "JPY")]

    def read(self, partition):
        rates = {"USD": 0.92, "GBP": 1.17, "JPY": 0.0061}  # stand-in for one API call per partition
        yield (partition.value, rates[partition.value])

spark.dataSource.register(FxRatesDataSource)
fx = spark.read.format("fx_rates").load()
fx.orderBy("currency").show()
print("partitions:", fx.rdd.getNumPartitions())

**What to notice:** `spark.read.format("fx_rates")` now works like a built-in format. Three `InputPartition`s became three partitions,
so a real source would make its three API calls in parallel on executors.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why is a Python UDF slower than a built-in function?

**A.** Rows are serialized from the JVM to a Python worker and back, and the optimizer cannot see inside the function, so it cannot push filters past it or generate code for it.

**Q2.** A UDF returns only nulls. What is the likely cause?

**A.** The declared return type does not match what the function returns; mismatched values become null silently.

**Q3.** What is the difference between `useArrow=True` and a pandas UDF?

**A.** Both use Arrow to move data, but an Arrow UDF still calls your function once per row, while a pandas UDF calls it once per batch with a whole Series, so the work itself is vectorized.

**Q4.** Why is `s.mean()` wrong inside a Series-to-Series pandas UDF used for z-scores?

**A.** The UDF sees one batch at a time, so the mean is the batch mean. Compute global statistics with an aggregate and join or broadcast them.

**Q5.** When do you use `applyInPandas` and when `mapInPandas`?

**A.** `applyInPandas` when the logic needs all rows of a group together (it shuffles by key). `mapInPandas` for batch-by-batch processing with no grouping and no shuffle.

**Q6.** Why trust accumulators only for monitoring?

**A.** Updates made inside transformations can be applied more than once when tasks are retried or stages recomputed.

## Key takeaways

- Prefer built-ins; every Python function crosses the JVM-Python boundary.
- Declare UDF return types exactly; mismatches become silent nulls.
- Use Arrow: `useArrow=True` for row UDFs, pandas UDFs for vectorized work.
- `applyInPandas` for per-group pandas logic, `mapInPandas` for per-batch logic, UDTFs for many rows per input.
- Bring only small results to pandas.

**Next:** Practise this module in [PySpark drills 9: Python UDFs and pandas](pyspark-drills-09-python-udfs-and-pandas.ipynb), then continue with [PySpark 10: batch pipelines](pyspark-10-batch-pipelines.ipynb).